# Imports


In [26]:
from __future__ import annotations

import gzip
import math
import copy
import struct
from pathlib import Path
from typing import Iterable

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split


# Constants


In [27]:
RANDOM_SEED = 42


In [28]:
DATA_DIR: Path = Path("/home/linkezio/Datasets/MNIST")


In [29]:
MODELS_DIR: Path = Path("/home/linkezio/Projects/Efficient-Polling-Based-Learning-Rate-Optimization-for-Neural-Networks/models")


In [30]:
EPOCHS = 20

# Configs


## Seeds


In [31]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## Device


In [32]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


device: cuda


# Data


## IDX helpers


In [33]:
# Opens a regular or gzipped binary file.
def _open_maybe_gzip(path: Path):
    return gzip.open(path, "rb") if path.suffix == ".gz" else path.open("rb")


# Reads an IDX/ubyte file and returns data in its original shape.
def _read_idx(path: Path) -> np.ndarray:
    with _open_maybe_gzip(path) as f:
        header = f.read(4)
        if len(header) != 4:
            raise ValueError(f"Invalid IDX file: incomplete header in {path}")

        zero_1, zero_2, data_type, dims = struct.unpack(">BBBB", header)
        if (zero_1, zero_2) != (0, 0):
            raise ValueError(f"Invalid IDX file: incorrect prefix in {path}")
        if data_type != 0x08:
            raise ValueError(f"Unsupported IDX type ({data_type}) em {path}")

        shape = tuple(struct.unpack(">I", f.read(4))[0] for _ in range(dims))
        data = f.read()

    arr = np.frombuffer(data, dtype=np.uint8)
    expected_size = int(np.prod(shape))
    if arr.size != expected_size:
        raise ValueError(
            f"Inconsistent IDX in {path}: expected {expected_size} elements, got {arr.size}"
        )
    return arr.reshape(shape)


# Automatically locates MNIST train/test files.
def find_mnist_files(data_dir: Path) -> dict[str, Path]:
    candidates = [p for p in data_dir.glob("*") if p.is_file()]
    names = {p.name: p for p in candidates}

    def pick(prefixes: Iterable[str]) -> Path:
        for p in candidates:
            lower = p.name.lower()
            if any(lower.startswith(pref) for pref in prefixes) and ("idx" in lower or "ubyte" in lower):
                return p
        raise FileNotFoundError(f"Could not find MNIST files in {data_dir}. Files found: {sorted(names)}")

    return {
        "train_images": pick(["train-images", "train_images", "train-images-idx3"]),
        "train_labels": pick(["train-labels", "train_labels", "train-labels-idx1"]),
        "test_images": pick(["t10k-images", "test-images", "t10k_images", "t10k-images-idx3"]),
        "test_labels": pick(["t10k-labels", "test-labels", "t10k_labels", "t10k-labels-idx1"]),
    }


## Dataset Class


In [34]:
class MNISTDataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        train: bool,
        mean: torch.Tensor | None = None,
        std: torch.Tensor | None = None,
    ):
        files = find_mnist_files(data_dir)
        if train:
            images_path, labels_path = files["train_images"], files["train_labels"]
        else:
            images_path, labels_path = files["test_images"], files["test_labels"]

        images = _read_idx(images_path)
        labels = _read_idx(labels_path)
        if images.ndim != 3:
            raise ValueError(f"Expected MNIST images (N,H,W). Received: {images.shape}")
        if labels.ndim != 1:
            raise ValueError(f"Labels MNIST expected (N,). Received: {labels.shape}")
        if images.shape[0] != labels.shape[0]:
            raise ValueError("Number of images != number of labels")

        self.images = torch.from_numpy(images).float().unsqueeze(1)  # (N,1,28,28), 0–255
        self.labels = torch.from_numpy(labels).long()

        if (mean is None) ^ (std is None):
            raise ValueError("Pass `mean` and `std` together, or neither.")
        self.mean = mean
        self.std = std

    # Returns the total number of dataset samples.
    def __len__(self) -> int:
        return int(self.labels.shape[0])

    # Returns one sample (x, y), with optional normalization.
    def __getitem__(self, idx: int):
        x = self.images[idx]
        if self.mean is not None:
            x = (x - self.mean) / self.std
        y = self.labels[idx]
        return x, y


## Calculate mean and std for normalize later


In [35]:

# Computes per-channel mean and standard deviation over the full dataset.
def compute_mean_std(dataset, batch_size=512):
    loader_mean = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    channel_sum = None
    n_pixels = 0
    for images, _ in loader_mean:
        b, c, h, w = images.shape
        if channel_sum is None:
            channel_sum = torch.zeros(c, dtype=torch.float64)
        channel_sum += images.double().sum(dim=(0, 2, 3))
        n_pixels += b * h * w

    mean = (channel_sum / n_pixels).float()

    loader_var = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sum_sq = None
    for images, _ in loader_var:
        b, c, h, w = images.shape
        if sum_sq is None:
            sum_sq = torch.zeros(c, dtype=torch.float64)
        diff = images.double() - mean.view(1, c, 1, 1).double()
        sum_sq += (diff * diff).sum(dim=(0, 2, 3))

    var = (sum_sq / n_pixels).float()
    std = torch.sqrt(var)
    std = torch.clamp(std, min=1e-8)
    return mean, std


In [36]:
train_for_stats = MNISTDataset(DATA_DIR, train=True)

mnist_mean, mnist_std = compute_mean_std(train_for_stats)

mnist_mean = mnist_mean.view(1, 1, 1)
mnist_std = mnist_std.view(1, 1, 1)

print("mean (1 channel):", mnist_mean.squeeze().tolist())
print("std  (1 channel):", mnist_std.squeeze().tolist())


mean (1 channel): 33.31842041015625
std  (1 channel): 78.56748962402344


## Train, Validation, Test Split


In [37]:
class DataLoaderHyperparameters:
    batch_size: int = 128
    val_fraction: float = 0.1
    num_workers: int = 0  # Jupyter: use 0 (workers cannot resolve classes in __main__).

data_loader_hyperparameters = DataLoaderHyperparameters()


In [38]:
full_train = MNISTDataset(DATA_DIR, train=True, mean=mnist_mean, std=mnist_std)
test_ds = MNISTDataset(DATA_DIR, train=False, mean=mnist_mean, std=mnist_std)

val_size = max(1, int(len(full_train) * data_loader_hyperparameters.val_fraction))
train_size = len(full_train) - val_size

train_ds, val_ds = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

train_loader = DataLoader(
    train_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=True,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)

len(train_ds), len(val_ds), len(test_ds)


(54000, 6000, 10000)

# Model


## Hyperparameters


In [39]:
class ModelHyperparameters:
    batch_size: int = 128
    epochs: int = EPOCHS
    lr: float = 1e-2  # Sec. 5.4 — ordinary ANN with ADAM (Model 4) uses alpha = 0.01
    weight_decay: float = 0.0
    num_workers: int = 0  # Jupyter

model_hyperparameters = ModelHyperparameters()


## Model Class


In [40]:
class SimpleMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    # Runs the model forward pass.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


# Train


## Metric Functions


In [41]:
# Computes average batch accuracy.
def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


In [42]:
loss_fn = nn.CrossEntropyLoss()


## Epoch Functions


In [43]:
@torch.inference_mode()
# Evaluates the model for one epoch and returns average loss/accuracy.
def eval_epoch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[float, float]:
    model.eval()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        losses.append(loss_fn(logits, y).item())
        accs.append(accuracy(logits, y))
    return float(np.mean(losses)), float(np.mean(accs))


# Trains the model for one epoch and returns average loss/accuracy.
def train_epoch(model: nn.Module, loader: DataLoader, optim: torch.optim.Optimizer, loss_fn: nn.Module) -> tuple[float, float]:
    model.train()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optim.step()

        losses.append(loss.item())
        accs.append(accuracy(logits.detach(), y))
    return float(np.mean(losses)), float(np.mean(accs))


import copy

# Polling Method + Adam (paper Sec. 3.6-3.8, Sec. 5.2 Model 2): one backward, n Adam steps with
# distinct learning rates; forward each candidate on the training minibatch; keep highest accuracy.
def train_epoch_polling(
    model: nn.Module,
    loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    candidate_lrs: tuple[float, ...],
) -> tuple[float, float, float]:
    model.train()
    losses: list[float] = []
    accs: list[float] = []
    chosen_lrs: list[float] = []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()

        best_acc = -1.0
        best_lr = candidate_lrs[0]
        best_snapshot: tuple[dict, dict] | None = None

        for lr in candidate_lrs:
            for pg in optim.param_groups:
                pg["lr"] = lr
            snap_m = copy.deepcopy(model.state_dict())
            snap_o = copy.deepcopy(optim.state_dict())
            optim.step()

            with torch.no_grad():
                logits_try = model(x)
                acc_try = accuracy(logits_try, y)

            model.load_state_dict(snap_m)
            optim.load_state_dict(snap_o)

            if acc_try > best_acc:
                best_acc = acc_try
                best_lr = lr
                best_snapshot = (snap_m, snap_o)

        assert best_snapshot is not None
        model.load_state_dict(best_snapshot[0])
        optim.load_state_dict(best_snapshot[1])
        for pg in optim.param_groups:
            pg["lr"] = best_lr
        optim.step()

        with torch.no_grad():
            logits_final = model(x)
            losses.append(loss_fn(logits_final, y).item())
            accs.append(accuracy(logits_final, y))
        chosen_lrs.append(best_lr)

    return (
        float(np.mean(losses)),
        float(np.mean(accs)),
        float(np.mean(chosen_lrs)),
    )



## Trainings

### Training X (Baseline)


In [44]:
# Coordinates epoch training/validation and saves the best checkpoint.
def fit_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    epochs: int,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    """Coordinates epoch training/validation and saves the best checkpoint."""
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(float(optim.param_groups[0]["lr"]))

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [45]:
model = SimpleMNISTCNN().to(DEVICE)
optim = torch.optim.Adam(
    model.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)

In [46]:
history, best_val_acc, model_path = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    model_path=MODELS_DIR / "mnist_best.pt",
)

epoch 01/20 | train loss 0.2114 acc 0.9363 | val loss 0.0779 acc 0.9785
epoch 02/20 | train loss 0.0760 acc 0.9764 | val loss 0.0757 acc 0.9797
epoch 03/20 | train loss 0.0631 acc 0.9811 | val loss 0.0696 acc 0.9772
epoch 04/20 | train loss 0.0532 acc 0.9835 | val loss 0.0699 acc 0.9807
epoch 05/20 | train loss 0.0521 acc 0.9839 | val loss 0.0663 acc 0.9813
epoch 06/20 | train loss 0.0513 acc 0.9845 | val loss 0.0738 acc 0.9817
epoch 07/20 | train loss 0.0469 acc 0.9864 | val loss 0.0941 acc 0.9770
epoch 08/20 | train loss 0.0472 acc 0.9859 | val loss 0.0667 acc 0.9830
epoch 09/20 | train loss 0.0431 acc 0.9874 | val loss 0.0965 acc 0.9793
epoch 10/20 | train loss 0.0460 acc 0.9869 | val loss 0.1116 acc 0.9742
epoch 11/20 | train loss 0.0485 acc 0.9864 | val loss 0.1128 acc 0.9747
epoch 12/20 | train loss 0.0414 acc 0.9879 | val loss 0.1056 acc 0.9780
epoch 13/20 | train loss 0.0381 acc 0.9890 | val loss 0.0758 acc 0.9830
epoch 14/20 | train loss 0.0397 acc 0.9884 | val loss 0.1016 acc

### Training Y (Library LR Scheduler)


In [47]:
class SchedulerHyperparameters:
    epochs: int = model_hyperparameters.epochs
    max_lr: float = model_hyperparameters.lr
    min_lr: float = model_hyperparameters.lr * 0.05

scheduler_hyperparameters = SchedulerHyperparameters()


In [48]:
# Trains with a built-in PyTorch scheduler (CosineAnnealingWarmRestarts).
def fit_model_with_library_scheduler(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    scheduler_hp: SchedulerHyperparameters,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optim,
        T_0=max(1, scheduler_hp.epochs // 4),
        T_mult=2,
        eta_min=scheduler_hp.min_lr,
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, scheduler_hp.epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        scheduler.step()
        current_lr = float(optim.param_groups[0]["lr"])

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(current_lr)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{scheduler_hp.epochs} | "
            f"lr {current_lr:.6f} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [49]:
model_scheduler = SimpleMNISTCNN().to(DEVICE)
optim_scheduler = torch.optim.Adam(
    model_scheduler.parameters(),
    lr=scheduler_hyperparameters.max_lr,
    weight_decay=model_hyperparameters.weight_decay,
)

In [50]:
history_scheduler, best_val_acc_scheduler, model_path_scheduler = fit_model_with_library_scheduler(
    model=model_scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim_scheduler,
    loss_fn=loss_fn,
    scheduler_hp=scheduler_hyperparameters,
    model_path=MODELS_DIR / "mnist_best_scheduler.pt",
)


epoch 01/20 | lr 0.009093 | train loss 0.3885 acc 0.8786 | val loss 0.2089 acc 0.9325
epoch 02/20 | lr 0.006718 | train loss 0.1571 acc 0.9493 | val loss 0.1361 acc 0.9597
epoch 03/20 | lr 0.003782 | train loss 0.1197 acc 0.9613 | val loss 0.1370 acc 0.9590
epoch 04/20 | lr 0.001407 | train loss 0.0844 acc 0.9722 | val loss 0.0926 acc 0.9703
epoch 05/20 | lr 0.010000 | train loss 0.0636 acc 0.9790 | val loss 0.0901 acc 0.9734
epoch 06/20 | lr 0.009768 | train loss 0.1385 acc 0.9559 | val loss 0.1669 acc 0.9489
epoch 07/20 | lr 0.009093 | train loss 0.1139 acc 0.9630 | val loss 0.1143 acc 0.9622
epoch 08/20 | lr 0.008042 | train loss 0.0950 acc 0.9694 | val loss 0.1148 acc 0.9661
epoch 09/20 | lr 0.006718 | train loss 0.0856 acc 0.9716 | val loss 0.1154 acc 0.9642
epoch 10/20 | lr 0.005250 | train loss 0.0783 acc 0.9748 | val loss 0.1001 acc 0.9722
epoch 11/20 | lr 0.003782 | train loss 0.0596 acc 0.9803 | val loss 0.0911 acc 0.9747
epoch 12/20 | lr 0.002458 | train loss 0.0501 acc 0.98

### Training Z (Polling Method)


In [51]:
class PollingHyperparameters:
    candidate_lrs: tuple[float, ...] = (
        float(model_hyperparameters.lr * 0.25),
        float(model_hyperparameters.lr * 0.5),
        float(model_hyperparameters.lr),
        float(model_hyperparameters.lr * 2.0),
        float(model_hyperparameters.lr * 4.0),
    )


polling_hyperparameters = PollingHyperparameters()


def fit_model_polling(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    epochs: int,
    candidate_lrs: tuple[float, ...],
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    history: dict[str, list[float]] = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, mean_lr = train_epoch_polling(
            model, train_loader, optim, loss_fn, candidate_lrs
        )
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(mean_lr)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"mean train-chosen lr {mean_lr:.6f} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [52]:
model_polling = SimpleMNISTCNN().to(DEVICE)
optim_polling = torch.optim.Adam(
    model_polling.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)


In [53]:
history_polling, best_val_acc_polling, model_path_polling = fit_model_polling(
    model=model_polling,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim_polling,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    candidate_lrs=polling_hyperparameters.candidate_lrs,
    model_path=MODELS_DIR / "mnist_best_polling.pt",
)


epoch 01/20 | mean train-chosen lr 0.011309 | train loss 0.1040 acc 0.9713 | val loss 0.1463 acc 0.9580
epoch 02/20 | mean train-chosen lr 0.019514 | train loss 0.0560 acc 0.9864 | val loss 0.1512 acc 0.9588
epoch 03/20 | mean train-chosen lr 0.022121 | train loss 0.0533 acc 0.9881 | val loss 0.1227 acc 0.9655
epoch 04/20 | mean train-chosen lr 0.025059 | train loss 0.0616 acc 0.9854 | val loss 0.1334 acc 0.9660
epoch 05/20 | mean train-chosen lr 0.021623 | train loss 0.0502 acc 0.9894 | val loss 0.1538 acc 0.9624
epoch 06/20 | mean train-chosen lr 0.023063 | train loss 0.0590 acc 0.9864 | val loss 0.2259 acc 0.9503
epoch 07/20 | mean train-chosen lr 0.021315 | train loss 0.0586 acc 0.9869 | val loss 0.1347 acc 0.9691
epoch 08/20 | mean train-chosen lr 0.018685 | train loss 0.0389 acc 0.9913 | val loss 0.1495 acc 0.9724
epoch 09/20 | mean train-chosen lr 0.018519 | train loss 0.0435 acc 0.9907 | val loss 0.2890 acc 0.9577
epoch 10/20 | mean train-chosen lr 0.019816 | train loss 0.0582 

## Training animation


In [54]:
# Animate training: compare one or more `history` dicts (train/val loss + optional lr).
# All histories must have the same epoch count. Columns follow list order (left → right).
#
# Example:
#   animate_training_compare(
#       [history_scheduler, history_polling],
#       titles=["Cosine warm restarts", "Polling + Adam"],
#   )

from IPython.display import HTML
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np

# Increase HTML animation budget (MB) to avoid dropped frames in long runs.
mpl.rcParams["animation.embed_limit"] = 80


def animate_training_compare(
    histories: list[dict],
    titles: list[str] | None = None,
    interval_ms: int = 80,
):
    if len(histories) < 1:
        raise ValueError("Pass a non-empty list of history dicts.")

    n = len(histories[0]["train_loss"])
    for i, h in enumerate(histories):
        if len(h["train_loss"]) != n:
            raise ValueError(
                f"All histories must have the same length (run 0 has {n}, run {i} has {len(h['train_loss'])})."
            )

    if titles is None:
        titles = [f"Run {j + 1}" for j in range(len(histories))]
    elif len(titles) != len(histories):
        raise ValueError("`titles` must have the same length as `histories`.")

    epochs = np.arange(1, n + 1)
    trains = [np.asarray(h["train_loss"], dtype=float) for h in histories]
    vals = [np.asarray(h["val_loss"], dtype=float) for h in histories]
    lrs = [np.asarray(h.get("lr", [float("nan")] * n), dtype=float) for h in histories]

    n_cols = len(histories)
    fig_w = max(12.0, 4.0 * n_cols)
    if n_cols == 1:
        fig, axes = plt.subplots(2, 1, figsize=(6.0, 7), constrained_layout=True)
        axes_loss = [axes[0]]
        axes_lr = [axes[1]]
    else:
        fig, axes = plt.subplots(2, n_cols, figsize=(fig_w, 7), constrained_layout=True)
        axes_loss = list(axes[0])
        axes_lr = list(axes[1])

    lines_train, lines_val = [], []
    dots_train, dots_val = [], []
    lines_lr, dots_lr = [], []
    vlines_loss, vlines_lr = [], []

    for j in range(n_cols):
        ax = axes_loss[j]
        ax.set_title(titles[j] + " — loss")
        ax.set_xlabel("epoch")
        ax.set_ylabel("loss")
        ax.grid(True, alpha=0.3)
        (lt,) = ax.plot([], [], label="train", color="C0")
        (lv,) = ax.plot([], [], label="val", color="C1")
        dt = ax.scatter([], [], color="C0", s=40, zorder=5)
        dv = ax.scatter([], [], color="C1", s=40, zorder=5)
        ax.legend(loc="upper right")
        lines_train.append(lt)
        lines_val.append(lv)
        dots_train.append(dt)
        dots_val.append(dv)
        vlines_loss.append(ax.axvline(1, color="gray", ls="--", alpha=0.5))

    lr_colors = [f"C{(k % 8) + 2}" for k in range(n_cols)]
    for j in range(n_cols):
        ax = axes_lr[j]
        ax.set_title(titles[j] + " — learning rate")
        ax.set_xlabel("epoch")
        ax.set_ylabel("lr")
        ax.grid(True, alpha=0.3)
        c = lr_colors[j]
        (ll,) = ax.plot([], [], color=c)
        dl = ax.scatter([], [], color=c, s=40, zorder=5)
        lines_lr.append(ll)
        dots_lr.append(dl)
        vlines_lr.append(ax.axvline(1, color="gray", ls="--", alpha=0.5))

    def init():
        for ax in axes_loss + axes_lr:
            ax.set_xlim(0.5, n + 0.5)
        y0 = float(min(*(float(t.min()) for t in trains), *(float(v.min()) for v in vals)))
        y1 = float(max(*(float(t.max()) for t in trains), *(float(v.max()) for v in vals)))
        pad = 0.05 * (y1 - y0 + 1e-9)
        for ax in axes_loss:
            ax.set_ylim(y0 - pad, y1 + pad)
        lr_stack = np.concatenate(lrs)
        if not np.isfinite(lr_stack).any():
            lr_lo, lr_hi = 0.0, 1.0
        else:
            lr_lo = float(np.nanmin(lr_stack))
            lr_hi = float(np.nanmax(lr_stack))
        lr_pad = 0.05 * (lr_hi - lr_lo + 1e-12)
        for ax in axes_lr:
            ax.set_ylim(lr_lo - lr_pad, lr_hi + lr_pad)
        return []

    def update(k: int):
        k = int(k)
        sl = slice(0, k + 1)
        ex = epochs[sl]

        for j in range(n_cols):
            lines_train[j].set_data(ex, trains[j][sl])
            lines_val[j].set_data(ex, vals[j][sl])
            dots_train[j].set_offsets(np.c_[ex[-1:], trains[j][sl][-1:]])
            dots_val[j].set_offsets(np.c_[ex[-1:], vals[j][sl][-1:]])
            lines_lr[j].set_data(ex, lrs[j][sl])
            dots_lr[j].set_offsets(np.c_[ex[-1:], lrs[j][sl][-1:]])

        xcur = float(epochs[k])
        for vl in vlines_loss + vlines_lr:
            vl.set_xdata([xcur, xcur])

        return []

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=n,
        init_func=init,
        interval=interval_ms,
        blit=False,
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())



In [56]:
# Already returns IPython.display.HTML; do not wrap with HTML() again.
animate_training_compare(
    [history, history_scheduler, history_polling],
    titles=["Baseline (fixed LR)", "CosineAnnealingWarmRestarts", "Polling + Adam"],
)

# Test


In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "mnist_best.pt", map_location=DEVICE))

test_loss, test_acc = eval_epoch(model, test_loader, loss_fn)

print(f"test loss {test_loss:.4f} | test acc {test_acc:.4f}")
